# Analyzing the SIDER 4.1 data

+ http://sideeffects.embl.de/
+ http://thinklab.com/d/30#4

In [4]:
import csv
import gzip
import collections

In [5]:
import pandas
import requests

In [3]:
# Download SIDER data
base_url = 'http://sideeffects.embl.de/media/download/'
filenames = [
    'README',
    'meddra_all_indications.tsv.gz',
    'meddra_all_se.tsv.gz',
    'meddra_freq.tsv.gz',
]
for filename in filenames:
    ! wget --no-verbose --timestamping --directory-prefix download {base_url}/{filename}

! mv download/README download/README.txt

2024-08-29 13:05:12 URL:http://sideeffects.embl.de/media/download//README [3304/3304] -> "download/README" [1]


## STITCH to DrugBank mapping utilities

In [8]:
def stitch_flat_to_pubchem(cid):
    assert cid.startswith('CID')
    return int(cid[3:]) - 1e8

def stitch_stereo_to_pubchem(cid):
    assert cid.startswith('CID')
    return int(cid[3:])

## meddra_freq.tsv.gz

In [8]:
columns = [
    'stitch_id_flat',
    'stitch_id_sterio',
    'umls_cui_from_label',
    'placebo',
    'frequency',
    'lower',
    'upper',
    'meddra_type',
    'umls_cui_from_meddra',
    'side_effect_name',
]
freq_df = pandas.read_table('download/meddra_freq.tsv.gz', names=columns)
freq_df.head(2)

,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,placebo,frequency,lower,upper,meddra_type,umls_cui_from_meddra,side_effect_name
0,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,LLT,C0000737,Abdominal pain
1,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,PT,C0000737,Abdominal pain


## meddra_all_se.tsv.gz

In [13]:
def pubchem_id_to_drug_name(cid):
    import requests

    # cid = str(row['pubchem_id'])  # Replace with your CID
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
    
    response = requests.get(url)
    data = response.json()
    try:
        first_synonym = data['InformationList']['Information'][0]['Synonym'][0]
    except:
        return cid

    return first_synonym

In [10]:
columns = [
    'stitch_id_flat',
    'stitch_id_sterio',
    'umls_cui_from_label',
    'meddra_type',
    'umls_cui_from_meddra',
    'side_effect_name',
]
se_df = pandas.read_table('download/meddra_all_se.tsv.gz', names=columns)
se_df['pubchem_id'] = se_df.stitch_id_sterio.map(stitch_stereo_to_pubchem)
# se_df = drugbank_map_df.merge(se_df)
se_df.head(2)

,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name,pubchem_id
0,CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps,10917
1,CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain,10917


In [14]:
se_df['drugbank_id'] = se_df.pubchem_id.map(pubchem_id_to_drug_name)
se_df

ReadTimeout: HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Read timed out. (read timeout=None)

In [11]:
se_df = se_df[['drugbank_id', 'umls_cui_from_meddra', 'side_effect_name']]
se_df = se_df.dropna()
se_df = se_df.drop_duplicates(['drugbank_id', 'umls_cui_from_meddra'])

# se_df = drugbank_df.merge(se_df)
se_df = se_df.sort_values(['drugbank_name', 'side_effect_name'])
# len(se_df)

KeyError: "['drugbank_id'] not in index"

In [11]:
# Create a reference of side effect IDs and Names
se_terms_df = se_df[['umls_cui_from_meddra', 'side_effect_name']].drop_duplicates()
assert se_terms_df.side_effect_name.duplicated().sum() == 0
se_terms_df = se_terms_df.sort_values('side_effect_name')
se_terms_df.to_csv('data/side-effect-terms.tsv', sep='\t', index=False)

In [12]:
# Side effects of cocaine
se_df.query("drugbank_id == 'DB00907'")

,drugbank_id,drugbank_name,umls_cui_from_meddra,side_effect_name
80494,DB00907,Cocaine,C0085631,Agitation
80495,DB00907,Cocaine,C0233571,Excitement
80486,DB00907,Cocaine,C0014549,Grand mal convulsion
80487,DB00907,Cocaine,C0020517,Hypersensitivity
80488,DB00907,Cocaine,C0026961,Mydriasis
80489,DB00907,Cocaine,C0027769,Nervousness
80496,DB00907,Cocaine,C1145670,Respiratory failure
80497,DB00907,Cocaine,C1325847,Sensitisation
80490,DB00907,Cocaine,C0233494,Tension
80491,DB00907,Cocaine,C0040822,Tremor


In [13]:
# Number of drugbank drugs
se_df.drugbank_id.nunique()

1223

In [14]:
# Number of UMLS side effects
se_df.umls_cui_from_meddra.nunique()

5734

In [15]:
# Save side effects
se_df.to_csv('data/side-effects.tsv', sep='\t', index=False)

## meddra_all_indications.tsv.gz

In [16]:
columns = [
    'stitch_id_flat',
    'umls_cui_from_label',
    'method',
    'concept_name',
    'meddra_type',
    'umls_cui_from_meddra',
    'meddra_name',
]
indication_df = pandas.read_table('download/meddra_all_indications.tsv.gz', names=columns)
indication_df['pubchem_id'] = indication_df.stitch_id_flat.map(stitch_flat_to_pubchem)

In [17]:
indication_df = drugbank_df.merge(drugbank_map_df.merge(indication_df))
indication_df = indication_df.query("meddra_type == 'PT'")
indication_df.head(2)

,drugbank_id,drugbank_name,pubchem_id,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name
1,DB00014,Goserelin,47725,CID100047725,C0002871,text_mention,Anemia,PT,C0002871,Anaemia
3,DB00014,Goserelin,47725,CID100047725,C0006142,NLP_indication,Malignant neoplasm of breast,PT,C0006142,Breast cancer


In [18]:
# Multiple Sclerosis indications
indication_df.query("umls_cui_from_meddra == 'C0026769'").drugbank_name.tolist()

['Baclofen',
 'Betamethasone',
 'Carbamazepine',
 'Triamcinolone',
 'Prednisone',
 'Tizanidine',
 'Hydrocortisone',
 'Prednisolone',
 'Methylprednisolone',
 'Mitoxantrone',
 'Dantrolene',
 'Dexamethasone',
 'FTY 720',
 'Dalfampridine',
 '(11alpha,14beta)-11,17,21-trihydroxypregn-4-ene-3,20-dione',
 'Fingolimod']

In [19]:
# Save indications
indication_df.to_csv('data/indications.tsv', sep='\t', index=False)